# **Assignment 2 - Individual Checkpoint 2**

## Basic inofrmation

**GroupID:** A2-CC06-Zone2-Group4

**Driving Question:** Do they show variation in activity levels depending on their sleep/wake times or late-night activity?

**Personal Planning and Progress Github wiki page:**

https://github.sydney.edu.au/fguo9672/2025-DATA3406-CC06-Group4/wiki/Fengyuan-Guo-%E2%80%90-Personal-Planning-and-Progress/1be741b1cc79737fd2f6880f25bdf801cc7c6f26


## User selection

**Selected Users for Checkpoint1**

- User 1503960366: High activity user (avg 12,117 steps/day, evening peak at 6 PM)

- User 1624580081: Variable activity user (avg 5,744 steps/day, morning peak at 10 AM)

- User 1644430081: Moderate activity user (avg 7,283 steps/day, evening peak at 7 PM)

**Selected user for deep analysis**
- Selected User ID: 1624580081

**Reason:**

I chose User 1624580081 for this detailed analysis because they contain many specific patterns for exploration. This user has extreme variability in daily activity (standard deviation of 6,177 steps) with a extreme outlier day of 36,019 steps—six times their average daily activity. Then, they demonstrate clear morning peak activity at 10 AM and traditional sleep patterns (11 PM - 5 AM). The combination of high variability and clear patterns provides opportunities for analysis and visualization of sleep/wake time relationships with activity levels.

## Statement of help received

### Writing code
- Generative AI (Claude): Helping for code structuring (literature programming) and developing comprehensive research approaches. Especially using AI for improving plots from my original codes

### Debugging code
- Generative AI (Claude): Received suggestions in detecting issues, fixing problems and testing functions

### Interpreting data
- Team Members: Discussed findings with group members to ensure consistency

### Visualizing data
- Course Materials: Used resources and knowledge for visualizations from lectures and workshops

## Initial Assumption and Prediction

Based on my previous checkpoint1, I have several specific assumptions and predictions for this user's activity patterns.

**Sleep/Wake pattern assumption**

By user's traditional sleep pattern (11 PM - 5 AM), I predict a clear 6-hour sleep period with low activity during these hours, followed by morning activity. Then, I expect to observe fast increase in activity levels leading to peak, and finally gradually decreases throughout the afternoon and evening.

**Activity distribution prediction**

I assump that this user's overall activity percentage will be concentrated in specific time intervals rather than spread throughout the day. Therefore, there is a extreme outlier day (36,019 steps), might because euther a special event or potential data collection issues.

**Sleep/Wake-Activity relationship prediction**

As a clear morning active type, I predict this user will show:
- Higher activity in morning/early afternoon hours
-Regular morning activity onset despite day-to-day variation
- Clear relationship between sleep timing and next-day morning activity patterns


These assumptions will guide my analysis approach and help validate research question about sleep/wake-activity relationships.

## Data loading and processing

First of all, I want to load the datasets and any libraries I need to use. Then, I will makea simple data processing function that can analyze individual user.

In [ ]:
# Import required libraries for comprehensive analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta


daily_steps = pd.read_csv('dailySteps_merged.csv')
hourly_steps = pd.read_csv('hourlySteps_merged.csv')
minute_steps = pd.read_csv('minuteStepsWide_merged.csv')

Then, I will create a simple data analysis function that can process any user's data to get basic inofrmation. This function is the foundation of my whole deep analysis, which can be used for any individual user.

In [ ]:
def analyze_single_user(user_id, daily_data, hourly_data, minute_data):
    # Filter data for the specified user
    user_daily = daily_data[daily_data['Id'] == user_id].copy()
    user_hourly = hourly_data[hourly_data['Id'] == user_id].copy()
    user_minute = minute_data[minute_data['Id'] == user_id].copy()


    # Process daily data
    user_daily['ActivityDay'] = pd.to_datetime(user_daily['ActivityDay'])
    user_daily = user_daily.sort_values('ActivityDay').reset_index(drop=True)

    # Process hourly data
    user_hourly['DateTime'] = pd.to_datetime(user_hourly['ActivityHour'], format="%m/%d/%Y %I:%M:%S %p")

    user_hourly['Hour'] = user_hourly['DateTime'].dt.hour
    user_hourly['Date'] = user_hourly['DateTime'].dt.date
    user_hourly = user_hourly.sort_values('DateTime').reset_index(drop=True)

    # Process minute data (get step columns)
    step_columns = [col for col in user_minute.columns if col.startswith('Steps')]
    user_minute['ActivityHour'] = pd.to_datetime(user_minute['ActivityHour'],format="%m/%d/%Y %I:%M:%S %p")
    user_minute = user_minute.sort_values('ActivityHour').reset_index(drop=True)

    # Calculate comprehensive statistics
    daily_stats = {
        'total_days': len(user_daily),
        'avg_daily_steps': user_daily['StepTotal'].mean(),
        'std_daily_steps': user_daily['StepTotal'].std(),
        'min_daily_steps': user_daily['StepTotal'].min(),
        'max_daily_steps': user_daily['StepTotal'].max(),
        'date_range': (user_daily['ActivityDay'].min(), user_daily['ActivityDay'].max())
    }

    # Hourly pattern analysis
    hourly_avg = user_hourly.groupby('Hour')['StepTotal'].agg(['mean', 'std']).reset_index()
    peak_hour = hourly_avg.loc[hourly_avg['mean'].idxmax(), 'Hour']
    sleep_threshold = hourly_avg['mean'].quantile(0.25)
    likely_sleep_hours = hourly_avg[hourly_avg['mean'] < sleep_threshold]['Hour'].tolist()

    # Minute-level analysis
    all_minute_steps = user_minute[step_columns].values.flatten()
    valid_minute_steps = all_minute_steps[~pd.isna(all_minute_steps)]

    minute_stats = {
        'total_minutes': len(valid_minute_steps),
        'active_minutes': np.sum(valid_minute_steps > 0),
        'activity_rate': np.sum(valid_minute_steps > 0) / len(valid_minute_steps) * 100,
        'avg_steps_per_minute': np.mean(valid_minute_steps),
        'max_steps_per_minute': np.max(valid_minute_steps)
    }

    return {
        'user_id': user_id,
        'daily_data': user_daily,
        'hourly_data': user_hourly,
        'minute_data': user_minute,
        'step_columns': step_columns,
        'daily_stats': daily_stats,
        'hourly_patterns': hourly_avg,
        'minute_stats': minute_stats,
        'peak_hour': peak_hour,
        'likely_sleep_hours': likely_sleep_hours,
        'sleep_threshold': sleep_threshold
    }


TARGET_USER = 1624580081
user_analysis = analyze_single_user(TARGET_USER, daily_steps, hourly_steps, minute_steps)

By that function, I can process sepecific user's data. Which includes daily, hourly and minutely data and other metrics. This ensures I can continue analysis by focusing on individual user.

## Comprehensive Data Analysis
I will analyze the data of user 1624580081 in several specific fields, to help me understand and explore our driving problem.

### Daily Activity Analysis

Firstly, I want to explore the daily activity patterns of User 1624580081 in detail. There is 1 outlier I observed in Checkpoint 1 should carefully understand, and I will make some plots to analyze the whole period distribution.

In [ ]:
def visualize_daily_patterns(analysis_results):
    user_daily = analysis_results['daily_data']
    daily_stats = analysis_results['daily_stats']
    user_id = analysis_results['user_id']

    # Create dashboard
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Daily Activity Analysis - User {user_id}', fontsize=16, fontweight='bold')

    # 1. Step count time series plot
    axes[0,0].plot(user_daily['ActivityDay'], user_daily['StepTotal'],
                   marker='o', linewidth=2, markersize=4, color='steelblue')
    axes[0,0].axhline(y=daily_stats['avg_daily_steps'], color='red', linestyle='--',
                      label=f'Average: {daily_stats["avg_daily_steps"]:.0f} steps', alpha=0.7)
    axes[0,0].axhline(y=daily_stats['max_daily_steps'], color='orange', linestyle=':',
                      label=f'Maximum: {daily_stats["max_daily_steps"]:,.0f} steps', alpha=0.7)
    axes[0,0].set_title('Step Count Over Time')
    axes[0,0].set_xlabel('Date')
    axes[0,0].set_ylabel('Steps')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    axes[0,0].tick_params(axis='x', rotation=45)

    # 2. Distribution of daily step counts
    axes[0,1].hist(user_daily['StepTotal'], bins=15, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0,1].axvline(daily_stats['avg_daily_steps'], color='red', linestyle='--',
                      label=f'Mean: {daily_stats["avg_daily_steps"]:.0f}')
    axes[0,1].axvline(daily_stats['avg_daily_steps'] + daily_stats['std_daily_steps'],
                      color='orange', linestyle=':', label=f'SD: {daily_stats["avg_daily_steps"] + daily_stats["std_daily_steps"]:.0f}')
    axes[0,1].set_title('Distribution of Step Counts')
    axes[0,1].set_xlabel('Daily Steps')
    axes[0,1].set_ylabel('Frequency')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)

    # 3. Box plot
    box_data = [user_daily['StepTotal']]
    box_plot = axes[1,0].boxplot(box_data, patch_artist=True, labels=['Daily Steps'])
    box_plot['boxes'][0].set_facecolor('lightblue')
    axes[1,0].set_title('Activity Box Plot')
    axes[1,0].set_ylabel('Steps')
    axes[1,0].grid(True, alpha=0.3)

    # Add outlier
    q75, q25 = np.percentile(user_daily['StepTotal'], [75, 25])
    iqr = q75 - q25
    outlier_threshold = q75 + 1.5 * iqr
    outliers = user_daily[user_daily['StepTotal'] > outlier_threshold]

    if not outliers.empty:
        outlier_text = f"{len(outliers)} outlier day(s): {outliers['StepTotal'].max():,.0f} steps"
        axes[1,0].text(0.7, 0.95, outlier_text, transform=axes[1,0].transAxes,
                       bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7),
                       verticalalignment='top')

    # 4. Cumulative steps plot
    user_daily['CumulativeSteps'] = user_daily['StepTotal'].cumsum()
    axes[1,1].plot(user_daily['ActivityDay'], user_daily['CumulativeSteps'],
                   linewidth=2, color='green', marker='o', markersize=3)
    axes[1,1].set_title('Cumulative Step Count')
    axes[1,1].set_xlabel('Date')
    axes[1,1].set_ylabel('Cumulative Steps')
    axes[1,1].grid(True, alpha=0.3)
    axes[1,1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    return outliers


outliers = visualize_daily_patterns(user_analysis)

print(f"\\n DAILY ACTIVITY SUMMARY FOR USER {TARGET_USER}:")
print(f"Monitoring period: {user_analysis['daily_stats']['date_range'][0].strftime('%Y-%m-%d')} to {user_analysis['daily_stats']['date_range'][1].strftime('%Y-%m-%d')}")
print(f"Total days monitored: {user_analysis['daily_stats']['total_days']}")
print(f"Average daily steps: {user_analysis['daily_stats']['avg_daily_steps']:.0f} ± {user_analysis['daily_stats']['std_daily_steps']:.0f}")
print(f"Activity range: {user_analysis['daily_stats']['min_daily_steps']:,.0f} to {user_analysis['daily_stats']['max_daily_steps']:,.0f} steps")

if not outliers.empty:
    print(f"Outlier days identified: {len(outliers)}")
    for _, day in outliers.iterrows():
        print(f"  - {day['ActivityDay'].strftime('%Y-%m-%d')}: {day['StepTotal']:,.0f} steps")

The daily activity analysis shows some patterns in User 1624580081's behavior.
- The standard deviation of 6,177 steps represents substantial day-to-day variation, with the coefficient of variation (std/mean) of 107% indicating highly inconsistent daily activity levels. Which is because that extreme value.
- Excluding the outlier day, this user maintains relatively low but consistent daily activity levels.


### Hourly and Sleep/Wake Analysis

Since our diriving problem is about the relationship between activity level and sleep/wake times, understanding the hourly activity patterns is important.

In [ ]:
def analyze_time_patterns(analysis_results):
    hourly_data = analysis_results['hourly_data']
    hourly_patterns = analysis_results['hourly_patterns']
    user_id = analysis_results['user_id']
    peak_hour = analysis_results['peak_hour']
    sleep_threshold = analysis_results['sleep_threshold']
    likely_sleep_hours = analysis_results['likely_sleep_hours']

    # Create dashboard
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Hourly data Analysis - User {user_id}', fontsize=16, fontweight='bold')

    # 1. Average hourly activity pattern
    hours = hourly_patterns['Hour']
    mean_steps = hourly_patterns['mean']
    std_steps = hourly_patterns['std']

    axes[0,0].plot(hours, mean_steps, marker='o', linewidth=3, markersize=8, color='blue')
    axes[0,0].fill_between(hours, mean_steps - std_steps, mean_steps + std_steps,
                           alpha=0.3, color='lightblue', label='Standard Deviation')
    axes[0,0].axhline(y=sleep_threshold, color='red', linestyle='--',
                      label=f'Sleep Threshold ({sleep_threshold:.0f} steps)', alpha=0.8)
    axes[0,0].scatter([peak_hour], [mean_steps[peak_hour]], color='red', s=150,
                      label=f'Peak Hour: {peak_hour}:00', zorder=5)

    # Highlight sleep hours
    for hour in likely_sleep_hours:
        axes[0,0].axvspan(hour-0.5, hour+0.5, alpha=0.2, color='purple')

    axes[0,0].set_title('24-Hour Activity Profile')
    axes[0,0].set_xlabel('Hour of Day')
    axes[0,0].set_ylabel('Average Steps per Hour')
    axes[0,0].set_xticks(range(0, 24, 2))
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)

    # 2. Activity heatmap by day and hour
    pivot_data = hourly_data.pivot_table(values='StepTotal', index=hourly_data['DateTime'].dt.date, columns='Hour', fill_value=0)

    sns.heatmap(pivot_data, cmap='YlOrRd', cbar_kws={'label': 'Steps per Hour'},
                ax=axes[0,1], xticklabels=2)
    axes[0,1].set_title('Daily-Hourly Heatmap')
    axes[0,1].set_xlabel('Hour of Day')
    axes[0,1].set_ylabel('Date')
    axes[0,1].tick_params(axis='y', rotation=0)

    # 3. Sleep/wake pattern analysis
    sleep_hours_only = [h for h in likely_sleep_hours if h >= 22 or h <= 6]
    wake_transition = hourly_patterns[hourly_patterns['Hour'].isin([5, 6, 7, 8])]['mean']

    axes[1,0].bar(likely_sleep_hours,
                  [mean_steps[mean_steps.index[hours == h]].iloc[0] for h in likely_sleep_hours],
                  alpha=0.7, color='purple', label='Likely Sleep Hours')
    axes[1,0].axhline(y=sleep_threshold, color='red', linestyle='--', alpha=0.8)
    axes[1,0].set_title('Sleep Period Activity Levels')
    axes[1,0].set_xlabel('Hour')
    axes[1,0].set_ylabel('Average Steps')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)

    # 4. Classification visualization
    morning_hours = [6, 7, 8, 9, 10, 11]
    evening_hours = [18, 19, 20, 21, 22, 23]

    morning_avg = hourly_patterns[hourly_patterns['Hour'].isin(morning_hours)]['mean'].mean()
    evening_avg = hourly_patterns[hourly_patterns['Hour'].isin(evening_hours)]['mean'].mean()

    me_data = ['Morning (6AM-11AM)', 'Evening (6PM-11PM)']
    me_values = [morning_avg, evening_avg]

    bars = axes[1,1].bar(me_data, me_values, color=['gold', 'navy'], alpha=0.7)
    axes[1,1].set_title('Morning vs Evening Activity')
    axes[1,1].set_ylabel('Average Steps per Hour')
    axes[1,1].grid(True, alpha=0.3)

    # Add value labels on bars
    for bar, value in zip(bars, me_values):
        axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                       f'{value:.0f}', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Calculate metrics
    morning_evening_ratio = morning_avg / evening_avg if evening_avg > 0 else float('inf')

    return {
        'morning_avg': morning_avg,
        'evening_avg': evening_avg,
        'morning_evening_ratio': morning_evening_ratio,
        'sleep_hours': sleep_hours_only,
        'wake_transition_hour': 6 if 6 in likely_sleep_hours else 5
    }

# Analysis
time_analysis = analyze_time_patterns(user_analysis)

print(f"\\n Time PATTERN SUMMARY FOR USER {TARGET_USER}:")
print(f"Peak activity hour: {user_analysis['peak_hour']}:00 ({user_analysis['hourly_patterns'].loc[user_analysis['peak_hour'], 'mean']:.0f} steps/hour)")
print(f"Likely sleep hours: {user_analysis['likely_sleep_hours']}")
print(f"Morning activity average (6AM-11AM): {time_analysis['morning_avg']:.0f} steps/hour")
print(f"Evening activity average (6PM-11PM): {time_analysis['evening_avg']:.0f} steps/hour")
print(f"Morning/Evening ratio: {time_analysis['morning_evening_ratio']:.2f}")

classification = "Morning" if time_analysis['morning_evening_ratio'] > 1.5 else "Evening" if time_analysis['morning_evening_ratio'] < 0.67 else "Intermediate"
print(f"Chronotype classification: **{classification}**")

- The morning/evening activity ratio of 2.4 clearly classifies this user as a morning type, validating our initial assessment from Checkpoint 1.
- The user demonstrates consistent sleep timing (11 PM - 5 AM) with minimal activity during these hours, suggesting regular sleep habits.
- The morning peak followed by continue decline illustrates how sleep/wake affects activity distribution throughout the day, directly addressing our research question.

### High-Activity Day Analysis
The outlier day of 36,019 steps should special consideration and further research. Understanding this will help determine whether it represents real variation or data collection issues, and how outliers relate to the user's normal sleep/wake patterns.

In [ ]:
def analyze_outlier_day(analysis_results, outlier_days):
    outlier_date = outlier_days.iloc[0]['ActivityDay'].date()
    hourly_data = analysis_results['hourly_data']
    minute_data = analysis_results['minute_data']
    step_columns = analysis_results['step_columns']
    user_id = analysis_results['user_id']

    # Get outlier day hourly data
    outlier_hourly = hourly_data[hourly_data['Date'] == outlier_date].copy()

    # Get outlier day minute data
    outlier_minute = minute_data[minute_data['ActivityHour'].dt.date == outlier_date].copy()

    # median activity day
    daily_data = analysis_results['daily_data']
    median_steps = daily_data['StepTotal'].median()
    typical_day = daily_data.iloc[(daily_data['StepTotal'] - median_steps).abs().argsort()[:1]]
    typical_date = typical_day.iloc[0]['ActivityDay'].date()

    typical_hourly = hourly_data[hourly_data['Date'] == typical_date].copy()

    # Create dashboard
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Outlier Day Analysis - User {user_id}\\n{outlier_date} vs Typical Day ({typical_date})',
                 fontsize=16, fontweight='bold')

    # 1. Hourly comparison between outlier and typical day
    if not outlier_hourly.empty and not typical_hourly.empty:
        axes[0,0].plot(outlier_hourly['Hour'], outlier_hourly['StepTotal'],
                       marker='o', linewidth=3, markersize=8, color='red',
                       label=f'Outlier Day ({outlier_date})')
        axes[0,0].plot(typical_hourly['Hour'], typical_hourly['StepTotal'],
                       marker='s', linewidth=2, markersize=6, color='blue', alpha=0.7,
                       label=f'Typical Day ({typical_date})')

        axes[0,0].set_title('Outlier vs Typical Day')
        axes[0,0].set_xlabel('Hour of Day')
        axes[0,0].set_ylabel('Steps per Hour')
        axes[0,0].legend()
        axes[0,0].grid(True, alpha=0.3)
        axes[0,0].set_xticks(range(0, 24, 2))

    # 2. Outlier day activity distribution by hour
    if not outlier_hourly.empty:
        hourly_percentages = outlier_hourly['StepTotal'] / outlier_hourly['StepTotal'].sum() * 100

        bars = axes[0,1].bar(outlier_hourly['Hour'], hourly_percentages,
                             color='orange', alpha=0.7)
        axes[0,1].set_title(f'Outlier Activity Distribution')
        axes[0,1].set_xlabel('Hour of Day')
        axes[0,1].set_ylabel('Percentage of Daily Steps')
        axes[0,1].grid(True, alpha=0.3)
        axes[0,1].set_xticks(range(0, 24, 2))

        # Highlight peak hours
        peak_hours = hourly_percentages.nlargest(3)
        for hour, pct in peak_hours.items():
            axes[0,1].text(hour, pct + 0.5, f'{pct:.1f}%',
                           ha='center', va='bottom', fontweight='bold')

    # 3. Minute-level analysis for outlier day peak hours
    if not outlier_minute.empty:
        # Find the peak activity hour
        peak_hour_data = outlier_hourly.loc[outlier_hourly['StepTotal'].idxmax()]
        peak_hour = int(peak_hour_data['Hour'])

        # Get minute data for that hour
        peak_minute_row = outlier_minute[outlier_minute['ActivityHour'].dt.hour == peak_hour]

        if not peak_minute_row.empty:
            minute_steps = peak_minute_row[step_columns].iloc[0].values
            minutes = range(60)

            axes[1,0].bar(minutes, minute_steps, alpha=0.7, color='red')
            axes[1,0].set_title(f'Minute Activity')
            axes[1,0].set_xlabel('Minute of Hour')
            axes[1,0].set_ylabel('Steps per Minute')
            axes[1,0].grid(True, alpha=0.3)

            # Add summary statistics
            max_minute = minute_steps.max()
            avg_minute = minute_steps.mean()
            active_minutes = np.sum(minute_steps > 0)

            stats_text = f'Max: {max_minute:.0f} steps/min\\nAvg: {avg_minute:.1f} steps/min\\nActive: {active_minutes}/60 min'
            axes[1,0].text(0.98, 0.98, stats_text, transform=axes[1,0].transAxes,
                           bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.8),
                           verticalalignment='top', horizontalalignment='right')

    # 4. Sleep pattern
    night_before_outlier = outlier_date - timedelta(days=1)
    night_before_hourly = hourly_data[hourly_data['Date'] == night_before_outlier]

    # Get late night hours (10PM-6AM) for both nights
    outlier_night_hours = [22, 23, 0, 1, 2, 3, 4, 5]
    if not night_before_hourly.empty:
        night_before_sleep = night_before_hourly[night_before_hourly['Hour'].isin(outlier_night_hours)]
        outlier_day_early = outlier_hourly[outlier_hourly['Hour'].isin([0, 1, 2, 3, 4, 5])]

        if not night_before_sleep.empty and not outlier_day_early.empty:
            combined_sleep_data = pd.concat([
                night_before_sleep[['Hour', 'StepTotal']].assign(Period='Night Before'),
                outlier_day_early[['Hour', 'StepTotal']].assign(Period='Outlier Day Early AM')
            ])

            sns.barplot(data=combined_sleep_data, x='Hour', y='StepTotal', hue='Period', ax=axes[1,1])
            axes[1,1].set_title('Sleep Pattern')
            axes[1,1].set_xlabel('Hour')
            axes[1,1].set_ylabel('Steps')
            axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Calculate outlier day statistics
    outlier_stats = {
        'total_steps': outlier_hourly['StepTotal'].sum() if not outlier_hourly.empty else 0,
        'peak_hour': peak_hour if not outlier_hourly.empty else None,
        'peak_hour_steps': outlier_hourly['StepTotal'].max() if not outlier_hourly.empty else 0,
        'active_hours': (outlier_hourly['StepTotal'] > 100).sum() if not outlier_hourly.empty else 0,
        'early_morning_activity': outlier_hourly[outlier_hourly['Hour'].isin([5, 6, 7, 8])]['StepTotal'].sum() if not outlier_hourly.empty else 0
    }

    return outlier_stats

# Analysis
outlier_investigation = analyze_outlier_day(user_analysis, outliers)

print(f"\\n OUTLIER DAY INVESTIGATION RESULTS:")
if outlier_investigation:
  print(f"• Date: {outlier_days.iloc[0]['ActivityDay'].strftime('%Y-%m-%d')}")
  print(f"• Total steps: {outlier_investigation['total_steps']:,.0f}")
  print(f"• Peak activity hour: {outlier_investigation['peak_hour']}:00")
  print(f"• Peak hour steps: {outlier_investigation['peak_hour_steps']:,.0f}")
  print(f"• Hours with >100 steps: {outlier_investigation['active_hours']}/24")
  print(f"• Early morning activity (5-8 AM): {outlier_investigation['early_morning_activity']:,.0f} steps")

- The outlier day shows continuous activity across 12 hours, with the highest single-hour count still following minute-by-minute patterns. This means rela variation rather than data collecting issue.
- Even on outlier day, the user maintained morning-active pattern, with highest activity occurring during 10 AM peak, showing the stability of sleep/wake patterns.
- The sleep pattern analysis shows small influence to sleep/wake periods before and during the high-activity day, the user maintained regular sleep schedule despite the extreme activity.

### Activity Level and Sleep/Wake Relationship Analysis
To fully explore our driving research question, I need to examine how activity level relates to their sleep/wake patterns and whether their late-night activity influences their activity pattern.

In [ ]:
def analyze_activity_efficiency(analysis_results):
    user_id = analysis_results['user_id']
    minute_data = analysis_results['minute_data']
    step_columns = analysis_results['step_columns']
    hourly_data = analysis_results['hourly_data']
    minute_stats = analysis_results['minute_stats']

    # Calculate activity efficiency by hour
    hourly_efficiency = []

    for hour in range(24):
        hour_data = minute_data[minute_data['ActivityHour'].dt.hour == hour]
        if not hour_data.empty:
            all_minutes = hour_data[step_columns].values.flatten()
            valid_minutes = all_minutes[~pd.isna(all_minutes)]
            if len(valid_minutes) > 0:
                active_pct = (np.sum(valid_minutes > 0) / len(valid_minutes)) * 100
                avg_when_active = np.mean(valid_minutes[valid_minutes > 0]) if np.sum(valid_minutes > 0) > 0 else 0
                hourly_efficiency.append({
                    'Hour': hour,
                    'ActivePercentage': active_pct,
                    'AvgStepsWhenActive': avg_when_active,
                    'TotalMinutes': len(valid_minutes)
                })

    efficiency_df = pd.DataFrame(hourly_efficiency)

    # Create dashboard
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Activity & Sleep/Wake Relationship - User {user_id}',
                 fontsize=16, fontweight='bold')

    # 1. Activity percentage by hour
    axes[0,0].plot(efficiency_df['Hour'], efficiency_df['ActivePercentage'],
                   marker='o', linewidth=2, markersize=6, color='green')
    axes[0,0].axhline(y=minute_stats['activity_rate'], color='red', linestyle='--',
                      label=f'Overall Average: {minute_stats["activity_rate"]:.1f}%', alpha=0.8)

    # Highlight sleep hours
    sleep_hours = [23, 0, 1, 2, 3, 4, 5]
    for hour in sleep_hours:
        axes[0,0].axvspan(hour-0.5, hour+0.5, alpha=0.2, color='purple')

    axes[0,0].set_title('Percentage of Minutes Active by Hour')
    axes[0,0].set_xlabel('Hour of Day')
    axes[0,0].set_ylabel('% of Minutes Active')
    axes[0,0].set_xticks(range(0, 24, 2))
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)

    # 2. Intensity when active by hour
    axes[0,1].plot(efficiency_df['Hour'], efficiency_df['AvgStepsWhenActive'],
                   marker='s', linewidth=2, markersize=6, color='orange')
    axes[0,1].set_title('Average Steps per Minute')
    axes[0,1].set_xlabel('Hour of Day')
    axes[0,1].set_ylabel('Steps')
    axes[0,1].set_xticks(range(0, 24, 2))
    axes[0,1].grid(True, alpha=0.3)

    # Highlight peak efficiency hours
    peak_efficiency_hour = efficiency_df.loc[efficiency_df['AvgStepsWhenActive'].idxmax(), 'Hour']
    peak_efficiency_value = efficiency_df['AvgStepsWhenActive'].max()
    axes[0,1].scatter([peak_efficiency_hour], [peak_efficiency_value],
                      color='red', s=100, zorder=5, label=f'Peak: {peak_efficiency_hour}:00')
    axes[0,1].legend()

    # 3. Sleep activity levels
    sleep_data = hourly_data[hourly_data['Hour'].isin(sleep_hours)].copy()
    sleep_data['SleepQualityScore'] = 100 - (sleep_data['StepTotal'] / 10)  # Lower steps = better sleep
    sleep_data['SleepQualityScore'] = sleep_data['SleepQualityScore'].clip(0, 100)

    daily_sleep_quality = sleep_data.groupby('Date')['SleepQualityScore'].mean().reset_index()

    axes[1,0].plot(daily_sleep_quality['Date'], daily_sleep_quality['SleepQualityScore'],
                   marker='o', linewidth=2, color='blue')
    axes[1,0].set_title('Night Activity Analysis')
    axes[1,0].set_xlabel('Date')
    axes[1,0].set_ylabel('Sleep Quality Score')
    axes[1,0].tick_params(axis='x', rotation=45)
    axes[1,0].grid(True, alpha=0.3)

    # 4. Wake-up efficiency (morning activity ramp-up)
    morning_hours = [5, 6, 7, 8, 9, 10]
    wake_up_pattern = efficiency_df[efficiency_df['Hour'].isin(morning_hours)].copy()

    axes[1,1].bar(wake_up_pattern['Hour'], wake_up_pattern['ActivePercentage'],
                  alpha=0.7, color='gold')
    axes[1,1].set_title('Morning Activity Pattern')
    axes[1,1].set_xlabel('Hour')
    axes[1,1].set_ylabel('% Active Minutes')
    axes[1,1].grid(True, alpha=0.3)

    # Add efficiency scores on bars
    for _, row in wake_up_pattern.iterrows():
        axes[1,1].text(row['Hour'], row['ActivePercentage'] + 0.5,
                       f'{row["ActivePercentage"]:.1f}%',
                       ha='center', va='bottom', fontweight='bold', fontsize=9)

    plt.tight_layout()
    plt.show()

    # Calculate key metrics
    wake_hours = [6, 7, 8, 9, 10, 11]  # Traditional wake-up period
    wake_efficiency = efficiency_df[efficiency_df['Hour'].isin(wake_hours)]['ActivePercentage'].mean()

    sleep_hours_list = [23, 0, 1, 2, 3, 4, 5]
    sleep_disruption = efficiency_df[efficiency_df['Hour'].isin(sleep_hours_list)]['ActivePercentage'].mean()

    peak_activity_hour = efficiency_df.loc[efficiency_df['ActivePercentage'].idxmax(), 'Hour']
    peak_efficiency_time = efficiency_df.loc[efficiency_df['AvgStepsWhenActive'].idxmax(), 'Hour']

    return {
        'wake_efficiency': wake_efficiency,
        'sleep_disruption': sleep_disruption,
        'peak_activity_hour': peak_activity_hour,
        'peak_efficiency_hour': peak_efficiency_time,
        'overall_activity_rate': minute_stats['activity_rate'],
        'efficiency_data': efficiency_df
    }

# Analysis
efficiency_analysis = analyze_activity_efficiency(user_analysis)

print(f"\\n ACTIVITY EFFICIENCY ANALYSIS FOR USER {TARGET_USER}:")
print(f"Overall activity rate: {efficiency_analysis['overall_activity_rate']:.1f}% of time")
print(f"Wake-up efficiency (6-11 AM): {efficiency_analysis['wake_efficiency']:.1f}% active minutes")
print(f"Sleep disruption (11PM-5AM): {efficiency_analysis['sleep_disruption']:.1f}% active minutes")
print(f"Peak activity hour: {efficiency_analysis['peak_activity_hour']}:00")
print(f"Most efficient hour (highest steps/active minute): {efficiency_analysis['peak_efficiency_hour']}:00")

# Calculate efficiency score
score = efficiency_analysis['wake_efficiency'] / (efficiency_analysis['sleep_disruption'] + 0.1)
print(f"• Chronotype efficiency ratio: {score:.1f} (wake efficiency / sleep disruption)")

- The user shows 15.2% active minutes during wake-up hours (6-11 AM) compared to only 2.1% during sleep hours, yielding a ratio of 7.2, which strongly supports their classification as a morning active type.
- The night activity score shows relatively stable patterns with low night activity level, showing consistent sleep patterns that support their morning activity type.
- This user concentrates activity in specific morning hours, achieving higher step rates per active minute during after wake periods.

### average zero count for all users by hour
Next, we wnat to focus onindividual sleep/wakepatterns, so I made theline plots for each user by hour to measure their zero step count percentage to illustrate their sleep period.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

hourly_steps = pd.read_csv('hourlySteps_merged.csv')

# Convert ActivityHour to datetime
hourly_steps['DateTime'] = pd.to_datetime(hourly_steps['ActivityHour'], format="%m/%d/%Y %I:%M:%S %p")
hourly_steps['Hour'] = hourly_steps['DateTime'].dt.hour
hourly_steps['DayOfWeek'] = hourly_steps['DateTime'].dt.day_name()
hourly_steps['Date'] = hourly_steps['DateTime'].dt.date

# Get all unique user IDs
all_users = sorted(hourly_steps['Id'].unique())

# Create output directory for individual plots
output_dir = 'individual_zero_count_plots'
os.makedirs(output_dir, exist_ok=True)

def calculate_individual_zero_count(user_id, data):
    # Filter data for this user
    user_data = data[data['Id'] == user_id].copy()

    if user_data.empty:
        return None

    # Mark zero steps
    user_data['IsZero'] = (user_data['StepTotal'] == 0).astype(int)

    # Calculate zero count (proportion) for each hour-day combination
    zero_counts = []

    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

    for day_of_week in day_order:
        for hour in range(24):
            # Filter for this specific hour and day
            subset = user_data[
                (user_data['DayOfWeek'] == day_of_week) &
                (user_data['Hour'] == hour)
            ]

            if not subset.empty:
                # Count how many times this hour-day had zero steps
                zero_count = subset['IsZero'].sum()
                total_count = len(subset)
                zero_percentage = (zero_count / total_count) * 100

                zero_counts.append({
                    'Hour': hour,
                    'DayOfWeek': day_of_week,
                    'ZeroCount': zero_count,
                    'TotalOccurrences': total_count,
                    'ZeroPercentage': zero_percentage
                })

    return pd.DataFrame(zero_counts)

def plot_individual_zero_count(user_id, df, total_days):

    if df is None or df.empty:
        return

    # Set up the plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))
    fig.suptitle(f'Zero Step Analysis - User {user_id}', fontsize=18, fontweight='bold')

    # Define colors for days
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE']

    # Plot 1: Zero Count (absolute numbers)
    for day, color in zip(day_order, colors):
        day_data = df[df['DayOfWeek'] == day].sort_values('Hour')
        ax1.plot(day_data['Hour'], day_data['ZeroCount'],
                marker='o', linewidth=2.5, markersize=5,
                label=day, color=color)

    ax1.set_title('Number of Zero-Step Hours', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Hour of Day', fontsize=12)
    ax1.set_ylabel('Count of Zero-Step Occurrences', fontsize=12)
    ax1.set_xticks(range(0, 24, 2))
    ax1.legend(loc='best', fontsize=10, ncol=2)
    ax1.grid(True, alpha=0.3, linestyle='--')

    # Add shaded regions
    ax1.axvspan(0, 6, alpha=0.1, color='purple')
    ax1.axvspan(9, 17, alpha=0.1, color='green')

    # Plot 2: Zero Percentage
    for day, color in zip(day_order, colors):
        day_data = df[df['DayOfWeek'] == day].sort_values('Hour')
        ax2.plot(day_data['Hour'], day_data['ZeroPercentage'],
                marker='s', linewidth=2.5, markersize=5,
                label=day, color=color)

    ax2.set_title('Percentage of Time with Zero Steps', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Hour of Day', fontsize=12)
    ax2.set_ylabel('Zero-Step Percentage (%)', fontsize=12)
    ax2.set_xticks(range(0, 24, 2))
    ax2.set_ylim(0, 105)
    ax2.legend(loc='best', fontsize=10, ncol=2)
    ax2.grid(True, alpha=0.3, linestyle='--')

    # Add shaded regions
    ax2.axvspan(0, 6, alpha=0.1, color='purple')
    ax2.axvspan(9, 17, alpha=0.1, color='green')

    plt.tight_layout()

    # Save the plot
    filename = f'{output_dir}/user_{user_id}_zero_count.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

    return filename

def analyze_individual_patterns(user_id, df):

    if df is None or df.empty:
        return None

    # Calculate key metrics
    overall_zero_pct = df['ZeroPercentage'].mean()

    # Sleep hours (0-6 AM)
    sleep_hours = df[df['Hour'].isin(range(0, 7))]
    sleep_zero_pct = sleep_hours['ZeroPercentage'].mean()

    # Active hours (9-17)
    active_hours = df[df['Hour'].isin(range(9, 18))]
    active_zero_pct = active_hours['ZeroPercentage'].mean()

    # Weekday vs Weekend
    weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    weekends = ['Saturday', 'Sunday']

    weekday_zero_pct = df[df['DayOfWeek'].isin(weekdays)]['ZeroPercentage'].mean()
    weekend_zero_pct = df[df['DayOfWeek'].isin(weekends)]['ZeroPercentage'].mean()

    # Find peak sedentary time
    peak_sedentary = df.loc[df['ZeroPercentage'].idxmax()]

    # Find most active time
    most_active = df.loc[df['ZeroPercentage'].idxmin()]

    return {
        'user_id': user_id,
        'overall_zero_pct': overall_zero_pct,
        'sleep_zero_pct': sleep_zero_pct,
        'active_zero_pct': active_zero_pct,
        'weekday_zero_pct': weekday_zero_pct,
        'weekend_zero_pct': weekend_zero_pct,
        'peak_sedentary_hour': int(peak_sedentary['Hour']),
        'peak_sedentary_day': peak_sedentary['DayOfWeek'],
        'peak_sedentary_pct': peak_sedentary['ZeroPercentage'],
        'most_active_hour': int(most_active['Hour']),
        'most_active_day': most_active['DayOfWeek'],
        'most_active_pct': most_active['ZeroPercentage']
    }

all_summaries = []

for i, user_id in enumerate(all_users, 1):
    # Get user data
    user_df = hourly_steps[hourly_steps['Id'] == user_id]
    total_days = user_df['Date'].nunique()

    # Calculate zero counts
    zero_count_df = calculate_individual_zero_count(user_id, hourly_steps)

    if zero_count_df is not None and not zero_count_df.empty:
        # Create visualization
        filename = plot_individual_zero_count(user_id, zero_count_df, total_days)

        # Analyze patterns
        summary = analyze_individual_patterns(user_id, zero_count_df)
        if summary:
            all_summaries.append(summary)

summary_df = pd.DataFrame(all_summaries)

# Create a comparison visualization of all users
def create_comparison_plot(summary_df):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Zero Count Comparison Across All Participants', fontsize=18, fontweight='bold')

    # Sort by user_id for consistent display
    summary_df = summary_df.sort_values('user_id')

    # Plot 1: Overall zero percentage
    axes[0,0].barh(range(len(summary_df)), summary_df['overall_zero_pct'], color='steelblue', alpha=0.7)
    axes[0,0].set_yticks(range(len(summary_df)))
    axes[0,0].set_yticklabels([str(uid) for uid in summary_df['user_id']], fontsize=8)
    axes[0,0].set_xlabel('Overall Zero-Step Percentage (%)', fontsize=11)
    axes[0,0].set_ylabel('User ID', fontsize=11)
    axes[0,0].set_title('Overall Sedentary Rate', fontsize=12, fontweight='bold')
    axes[0,0].grid(True, alpha=0.3, axis='x')

    # Plot 2: Sleep vs Active hours zero percentage
    x = np.arange(len(summary_df))
    width = 0.35
    axes[0,1].barh(x - width/2, summary_df['sleep_zero_pct'], width, label='Sleep Hours (0-6 AM)', color='purple', alpha=0.7)
    axes[0,1].barh(x + width/2, summary_df['active_zero_pct'], width, label='Active Hours (9-17)', color='green', alpha=0.7)
    axes[0,1].set_yticks(x)
    axes[0,1].set_yticklabels([str(uid) for uid in summary_df['user_id']], fontsize=8)
    axes[0,1].set_xlabel('Zero-Step Percentage (%)', fontsize=11)
    axes[0,1].set_ylabel('User ID', fontsize=11)
    axes[0,1].set_title('Sleep vs Active Hours Comparison', fontsize=12, fontweight='bold')
    axes[0,1].legend(fontsize=9)
    axes[0,1].grid(True, alpha=0.3, axis='x')

    # Plot 3: Weekday vs Weekend
    x = np.arange(len(summary_df))
    axes[1,0].barh(x - width/2, summary_df['weekday_zero_pct'], width, label='Weekday', color='orange', alpha=0.7)
    axes[1,0].barh(x + width/2, summary_df['weekend_zero_pct'], width, label='Weekend', color='gold', alpha=0.7)
    axes[1,0].set_yticks(x)
    axes[1,0].set_yticklabels([str(uid) for uid in summary_df['user_id']], fontsize=8)
    axes[1,0].set_xlabel('Zero-Step Percentage (%)', fontsize=11)
    axes[1,0].set_ylabel('User ID', fontsize=11)
    axes[1,0].set_title('Weekday vs Weekend Comparison', fontsize=12, fontweight='bold')
    axes[1,0].legend(fontsize=9)
    axes[1,0].grid(True, alpha=0.3, axis='x')

    # Plot 4: Distribution of overall zero percentages
    axes[1,1].hist(summary_df['overall_zero_pct'], bins=15, color='steelblue', alpha=0.7, edgecolor='black')
    axes[1,1].axvline(summary_df['overall_zero_pct'].mean(), color='red', linestyle='--',
                      linewidth=2, label=f"Mean: {summary_df['overall_zero_pct'].mean():.1f}%")
    axes[1,1].set_xlabel('Overall Zero-Step Percentage (%)', fontsize=11)
    axes[1,1].set_ylabel('Number of Participants', fontsize=11)
    axes[1,1].set_title('Distribution of Sedentary Rates', fontsize=12, fontweight='bold')
    axes[1,1].legend(fontsize=9)
    axes[1,1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f'{output_dir}/all_users_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()

# Create comparison visualization
create_comparison_plot(summary_df)

Based on these plots, we can find their sleep patterns and their late-activity by each specific user. Then further research on our driving problem.

## Summary and conclusion

### Final Conclusion

- This data analysis of User 1624580081 provides detailed evidence that individuals will show significant variation in activity levels depending on their sleep/wake times and late-night activity patterns. The user's strong morning active type, with a 2.4:1 morning-to-evening activity ratio and concentrated activity during after wake periods, directly support our research hypothesis.

### Validation of Initial Predictions

- Confirmed with clear 10 AM peak and fast increasing in morning
- Validated through consistent 11 PM - 5 AM low-activity (sleep) periods
- Confirmed with higher efficiency during after wake time periods
- Outlier day maintained morning peak timing, showing stable activity pattern

### Research Question Answer
**Do they show variation in activity levels depending on their sleep/wake times or late-night activity?**

YES - User 1624580081 shows clear variations in activity levels based on sleep/wake timing, with a 7.2:1 efficiency ratio between wake and sleep periods, concentrated morning activity aligned with morning active type, and consistent low late-night activity supporting traditional sleep patterns.

### Summary
This individual checkpoint provides a detailed analysis of how sleep/wake type influences activity distribution, leding us to explore our driving problem in right way.